In [ ]:
# ======================================================
# Notebook: 2D contamination detection using Neural Net
# Bayesian-style sampling via uncertainty
# ======================================================

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (N,2)
y = np.load("/mnt/data/initial_outputs.npy")     # (N,)

# Binary labels (non-zero = contamination)
y_bin = (y > 0).astype(np.float32)

# Convert to torch tensors
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y_bin.reshape(-1,1), dtype=torch.float32)

# Simple neural network
model = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(32, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Train
model.train()
for epoch in range(500):
    optimizer.zero_grad()
    preds = model(X_t)
    loss = criterion(preds, y_t)
    loss.backward()
    optimizer.step()

# Candidate grid
x_min, x_max = X[:,0].min(), X[:,0].max()
y_min, y_max = X[:,1].min(), X[:,1].max()

gx = np.linspace(x_min, x_max, 100)
gy = np.linspace(y_min, y_max, 100)
XX, YY = np.meshgrid(gx, gy)
X_grid = np.vstack([XX.ravel(), YY.ravel()]).T
X_grid_t = torch.tensor(X_grid, dtype=torch.float32)

# MC Dropout for uncertainty (Bayesian-style)
model.train()  # keep dropout active
mc_samples = []

with torch.no_grad():
    for _ in range(30):
        mc_samples.append(model(X_grid_t).numpy())

mc_samples = np.stack(mc_samples)
mean_prob = mc_samples.mean(axis=0).flatten()
uncertainty = mc_samples.std(axis=0).flatten()

# Acquisition = uncertainty (exploration)
top_idx = np.argsort(uncertainty)[-10:]
next_points = X_grid[top_idx]

print("Next (10,2) inputs:")
print(next_points)

# Plot
plt.figure(figsize=(6,5))
plt.contourf(XX, YY, mean_prob.reshape(XX.shape), levels=30)
plt.scatter(X[:,0], X[:,1])
plt.scatter(next_points[:,0], next_points[:,1], marker='x', s=100)
plt.show()